## **Data cleaning pipeline**

### *Clinical Trial CTN-0051: Extended-Release Naltroxone vs. Buprenorphine for Opioid Treatment*




In [1]:
import pandas as pd
import numpy as np

### **1. Baseline data**
#### a) *DXS.csv - Detoxification Utilization Summary*
#### b) *dsm.csv - DSM-5 Substance Use Disorder criteria*
#### c) *ase.csv - ASI Employment / Economic module*



| File | CRF module | Content |
|------|------------|---------|
| `DXS.csv` | Detoxification Utilization Summary | Detox stay, last opioid, route, LOS |
| `dsm.csv` (inside `AE.xlsx`) | DSM-5 Substance Use Disorder | Poly-substance severity scores at baseline |
| `ase.csv` (inside `AE.xlsx`) | ASI-Lite Employment/Economic | Employment, income, education at baseline |

**Output:** `baseline_merged.csv` — one row per patient (PATID), 658 rows expected.

---
**Key conventions in this dataset:**
- All dates are integers = days relative to **randomisation day 0** (negative = before randomisation)
- `PATID` is the anonymous participant ID and the join key across all files
- `PROTSEG = 'A'` marks the baseline/screening phase; `'B'` marks the treatment phase
- `SITE` is always `NaN` in this dataset (redacted for de-identification)

In [2]:

PATH_DXS = "../raw-data/DXS.csv"
PATH_AE  = "../raw-data/AE.xlsx"    

# ── Output path ───────────────────────────────────────────────────────────────
PATH_OUT = "baseline_merged.csv"

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print("Imports OK")

Imports OK


### 1a. DXS — Detoxification Utilization Summary

One row per participant. Records the inpatient detox stay that preceded trial enrolment: admission and discharge dates, the last opioid used, its route, and whether the patient transferred to an outpatient or residential facility afterwards.

This is the **backbone** file — all 658 randomised participants appear here, including those who never started treatment.

In [3]:
dxs_raw = pd.read_csv(PATH_DXS)

print(f"DXS shape: {dxs_raw.shape}")
print(f"Unique PATIDs: {dxs_raw['PATID'].nunique()}")
print(f"Duplicate PATIDs: {dxs_raw['PATID'].duplicated().sum()}")
print()
dxs_raw.head(20)

DXS shape: (658, 17)
Unique PATIDs: 658
Duplicate PATIDs: 0



,PROT,PATID,SITE,RANDDT,PROTSEG,VISNO,DXLSOPDT,DXLSOPTM,DXLSOPSB,DXLSOPRT,DXLSTOPI,DXADMNDT,DXADMNTM,DXDDCDT,DXDCFCLT,DXFCDCDT,DXSCOMM
0,51,948442,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-10.0,09:00,4.0,0.0,NaN,NaN
1,51,85038,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-3.0,14:00,15.0,0.0,NaN,NaN
2,51,626492,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-6.0,14:00,2.0,0.0,NaN,NaN
3,51,632936,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-4.0,11:00,4.0,0.0,NaN,NaN
4,51,415576,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-1.0,11:00,8.0,0.0,NaN,NaN
5,51,471334,NaN,0.0,A,0,0.0,08:00,5.0,1.0,NaN,-3.0,12:00,0.0,0.0,NaN,NaN
6,51,626802,NaN,0.0,A,0,-1.0,08:00,5.0,1.0,NaN,-1.0,13:30,12.0,0.0,NaN,NaN
7,51,422129,NaN,0.0,A,0,0.0,12:00,5.0,1.0,NaN,-6.0,11:00,6.0,0.0,NaN,NaN
8,51,29321,NaN,0.0,A,0,0.0,12:00,5.0,1.0,NaN,-12.0,14:00,2.0,0.0,NaN,NaN
9,51,496667,NaN,0.0,A,0,0.0,08:00,5.0,1.0,1.0,-11.0,11:00,-2.0,0.0,NaN,NaN


### 1b. DSM — DSM-5 Substance Use Disorder criteria

One row per participant, baseline only (`PROTSEG = 'A'`, `VISNO = 0`). Contains binary items for 6 substances (opioids, alcohol, amphetamines, cannabis, cocaine, sedatives), each assessed against the 11 DSM-5 SUD criteria. The final columns (`DS*SCO`) are summed severity scores (0–4 scale).

In [4]:
dsm_raw = pd.read_excel(PATH_AE, sheet_name="dsm.csv")

print(f"DSM shape: {dsm_raw.shape}")
print(f"Unique PATIDs: {dsm_raw['PATID'].nunique()}")
print(f"PROTSEG values: {dsm_raw['PROTSEG'].unique()}")
print(f"VISNO values:   {dsm_raw['VISNO'].unique()}")
print()
dsm_raw.head(3)

DSM shape: (668, 86)
Unique PATIDs: 668
PROTSEG values: ['A']
VISNO values:   [0]



,PROT,PATID,SITE,RANDDT,PROTSEG,VISNO,DSMASMDT,DSOPI12M,DSALC12M,DSAMP12M,DSTHC12M,DSCOC12M,DSSED12M,DSOPIOBL,DSALCOBL,DSAMPOBL,DSTHCOBL,DSCOCOBL,DSSEDOBL,DSOPIHAZ,DSALCHAZ,DSAMPHAZ,DSTHCHAZ,DSCOCHAZ,DSSEDHAZ,DSOPISOC,DSALCSOC,DSAMPSOC,DSTHCSOC,DSCOCSOC,DSSEDSOC,DSOPITOL,DSALCTOL,DSAMPTOL,DSTHCTOL,DSCOCTOL,DSSEDTOL,DSOPIWIT,DSALCWIT,DSAMPWIT,DSTHCWIT,DSCOCWIT,DSSEDWIT,DSOPIDOS,DSALCDOS,DSAMPDOS,DSTHCDOS,DSCOCDOS,DSSEDDOS,DSOPICUT,DSALCCUT,DSAMPCUT,DSTHCCUT,DSCOCCUT,DSSEDCUT,DSOPITIM,DSALCTIM,DSAMPTIM,DSTHCTIM,DSCOCTIM,DSSEDTIM,DSOPIACT,DSALCACT,DSAMPACT,DSTHCACT,DSCOCACT,DSSEDACT,DSOPICON,DSALCCON,DSAMPCON,DSTHCCON,DSCOCCON,DSSEDCON,DSOPICRA,DSALCCRA,DSAMPCRA,DSTHCCRA,DSCOCCRA,DSSEDCRA,DSOPISCO,DSALCSCO,DSAMPSCO,DSTHCSCO,DSCOCSCO,DSSEDSCO,DSMCOMM
0,51,948442,NaN,0.0,A,0,-1.0,1,1,0,1,1,1,0,1.0,NaN,0.0,0.0,1.0,0,1.0,NaN,0.0,0.0,1.0,1,1.0,NaN,1.0,1.0,1.0,1,1.0,NaN,1.0,0.0,1.0,1,0.0,NaN,0.0,0.0,1.0,1,0.0,NaN,0.0,1.0,0.0,1,0.0,NaN,0.0,0.0,0.0,1,0.0,NaN,0.0,0.0,0.0,1,0.0,NaN,0.0,1.0,0.0,1,0.0,NaN,0.0,1.0,1.0,1,0.0,NaN,0.0,1.0,1.0,1,2.0,4.0,3.0,2.0,1.0,NaN
1,51,85038,NaN,0.0,A,0,-1.0,1,1,0,1,1,1,1,0.0,NaN,0.0,0.0,0.0,1,1.0,NaN,1.0,1.0,0.0,1,1.0,NaN,1.0,1.0,1.0,1,0.0,NaN,0.0,1.0,0.0,1,0.0,NaN,0.0,0.0,0.0,1,0.0,NaN,0.0,0.0,0.0,1,0.0,NaN,0.0,0.0,0.0,1,0.0,NaN,0.0,1.0,0.0,1,1.0,NaN,0.0,1.0,0.0,1,1.0,NaN,1.0,1.0,1.0,1,1.0,NaN,0.0,1.0,0.0,1,2.0,4.0,3.0,1.0,3.0,NaN
2,51,626492,NaN,0.0,A,0,0.0,1,0,0,1,0,0,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,0.0,NaN,NaN,1,4.0,4.0,4.0,4.0,4.0,NaN


### 1c. ASE — ASI-Lite Employment and Economic module

Multiple rows per participant across 4 timepoints (`VISNO`: `'00'` = baseline, `'24'` = end of treatment, `'EOT'` = early termination, `'M3FU'` = 3-month follow-up). We extract only the **baseline visit** (`VISNO = '00'`) to stay consistent with the other two files.

In [5]:
ase_raw = pd.read_excel(PATH_AE, sheet_name="ase.csv")

print(f"ASE shape (all timepoints): {ase_raw.shape}")
print(f"Unique PATIDs (all):        {ase_raw['PATID'].nunique()}")
print(f"VISNO distribution:")
print(ase_raw['VISNO'].value_counts().to_string())
print()
ase_raw.head(500)

ASE shape (all timepoints): (1592, 65)
Unique PATIDs (all):        623
VISNO distribution:
VISNO
00      623
M3FU    424
24      369
EOT     176



,PROT,PATID,SITE,RANDDT,PROTSEG,VISNO,AEEDCPYR,AEEDCPMT,AEEDCPNA,AEEDCPCM,AETECPMT,AETECPNA,AETECPCM,AEDRVLSC,AEDRVLCM,AEAUTOAV,AEAUTOCM,AEJOBYR,AEJOBMT,AEJOBNA,AEJOBCM,AEOCCUPT,AEOCCPSP,AEOCCPNA,AEOCCPCM,AESUPPRT,AESUPPCM,AEUSEMPL,AEUSEMCM,AEPAID,AEPAIDNA,AEPAIDCM,AEEMPMNY,AEEMNYNA,AEEMNYCM,AEUNEMNY,AEUMNYNA,AEUMNYCM,AEWLFMNY,AEWMNYNA,AEWMNYCM,AEPENMNY,AEPMNYNA,AEPMNYCM,AEMATMNY,AEMMNYNA,AEMMNYCM,AEILLMNY,AEIMNYNA,AEIMNYCM,AEDEPEND,AEDPNDNA,AEDPNDCM,AEEP30D,AEEP30NA,AEEP30CM,AEEBP30D,AEEB30NA,AEEB30CM,AEECI30D,AEEC30NA,AEEC30CM,AEMISREP,AEUNDRST,ASECOMM
0,51,948442,NaN,0.0,B,24,0,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,NaN,NaN,30.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,0,NaN
1,51,948442,NaN,0.0,A,00,10,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,2.0,6.0,NaN,NaN,7.0,NaN,NaN,NaN,0,NaN,3.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,213.0,NaN,NaN,0.0,NaN,NaN,10.0,NaN,NaN,300.0,NaN,NaN,0,NaN,NaN,30.0,NaN,NaN,3.0,NaN,NaN,1.0,NaN,NaN,0,0,NaN
2,51,85038,NaN,0.0,B,M3FU,0,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,0,NaN,NaN,NaN,22.0,NaN,NaN,1200.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,4,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,0,NaN
3,51,85038,NaN,0.0,A,00,12,0,NaN,NaN,48.0,NaN,NaN,0,NaN,0,NaN,2.0,0.0,NaN,NaN,5.0,NaN,NaN,NaN,1,NaN,7.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,600.0,NaN,NaN,0,NaN,NaN,30.0,NaN,NaN,3.0,NaN,NaN,2.0,NaN,NaN,0,0,NaN
4,51,626492,NaN,0.0,B,M3FU,0,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,NaN,1,NaN,NaN,NaN,23.0,NaN,NaN,1000.0,NaN,NaN,0.0,NaN,NaN,267.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,4,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,51,780180,NaN,0.0,A,00,14,0,NaN,NaN,0.0,NaN,NaN,1,NaN,1,NaN,6.0,0.0,NaN,NaN,3.0,NaN,NaN,NaN,1,NaN,2.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,350.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,1,NaN,NaN,30.0,NaN,NaN,4.0,NaN,NaN,4.0,NaN,NaN,0,0,NaN
496,51,495148,NaN,0.0,B,EOT,0,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,0,NaN,NaN,NaN,22.0,NaN,NaN,700.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,0,NaN
497,51,495148,NaN,0.0,A,00,13,0,NaN,NaN,8.0,NaN,NaN,0,NaN,0,NaN,0.0,8.0,NaN,NaN,6.0,NaN,NaN,NaN,0,NaN,3.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0,0,NaN
498,51,990538,NaN,0.0,B,M3FU,0,0,NaN,NaN,0.0,NaN,NaN,0,NaN,0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,0,NaN,NaN,NaN,20.0,NaN,NaN,NaN,97.0,NaN,0.0,NaN,NaN,NaN,97.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,4,NaN,NaN,NaN,96.0,NaN,NaN,96.0,NaN,0.0,NaN,NaN,0,0,NaN


## 2. Clean and select variables

### 2a. DXS 

We drop administrative columns (`PROT`, `SITE`, `RANDDT`, `DXSCOMM`) that carry no analytic information, derive the length of stay in detox (`detox_los`), and add human-readable label columns for the two key categoricals.

In [6]:
# ── Drop administrative / always-constant columns ─────────────────────────────
admin_cols = ['PROT', 'SITE', 'RANDDT', 'PROTSEG', 'VISNO', 'DXSCOMM']
dxs = dxs_raw.drop(columns=[c for c in admin_cols if c in dxs_raw.columns]).copy()

# ── Drop time-of-day string columns (no analytic value) ──────────────────────
time_cols = ['DXLSOPTM', 'DXADMNTM']
# time_cols = [c for c in time_cols if c in dxs.columns]
dxs = dxs.drop(columns=time_cols)
print(f"Dropped: {time_cols}")
print(f"DXS final shape: {dxs.shape}")

# ── Derived: length of stay in detox (days) ───────────────────────────────────
# DXADMNDT and DXDDCDT are integers (days relative to randomisation day 0)
# Negative values = event occurred before randomisation
dxs['detox_los'] = dxs['DXDDCDT'] - dxs['DXADMNDT']

# ── Label: last opioid substance ──────────────────────────────────────────────
substance_map = {
    1:  'Methadone',
    2:  'Oxycodone',
    3:  'Hydrocodone',
    4:  'Heroin',
    5:  'Non-study buprenorphine',
    6:  'Hydromorphone',
    7:  'Oxymorphone',
    8:  'Meperidine',
    9:  'Codeine',
    10: 'Fentanyl',
    11: 'Morphine',
}
dxs['last_substance_label'] = dxs['DXLSOPSB'].map(substance_map)

# ── Label: route of administration ────────────────────────────────────────────
route_map = {
    1: 'Oral',
    2: 'Nasal',
    3: 'Smoking',
    4: 'Non-IV injection',
    5: 'IV injection',
}
dxs['last_route_label'] = dxs['DXLSOPRT'].map(route_map)

# ── Label: discharge facility type ───────────────────────────────────────────
dxs['discharge_facility_label'] = dxs['DXDCFCLT'].map({0: 'Outpatient', 1: 'Residential'})

#RENAMING
# ── Rename remaining DXS columns to human-readable names ─────────────────────
dxs_rename = {
    'DXLSOPDT' : 'last_opioid_date',        # day of last opioid use (days from randomisation)
    'DXLSOPTM' : 'last_opioid_time',         # time of last opioid use (HH:MM string)
    'DXLSTOPI' : 'last_opioid_prior_to_rand',# flag: last use was prior to randomisation (1=yes)
    'DXADMNDT' : 'detox_admission_day',      # detox admission day (days from randomisation, negative)
    'DXADMNTM' : 'detox_admission_time',     # detox admission time (HH:MM string)
    'DXDDCDT'  : 'detox_discharge_day',      # detox discharge day (days from randomisation)
    'DXFCDCDT' : 'followup_care_start_day',  # day patient started follow-up care facility
    'DXLSOPSB' : 'last_substance_code',      # last opioid substance (numeric code — see label col)
    'DXLSOPRT' : 'last_route_code',          # route of administration (numeric code — see label col)
    'DXDCFCLT' : 'discharge_facility_code',  # discharge facility type (0=outpatient, 1=residential)
}
dxs = dxs.rename(columns={k: v for k, v in dxs_rename.items() if k in dxs.columns})

print("DXS columns after rename:")
print(dxs.columns.tolist())

print(f"DXS clean shape: {dxs.shape}")
print(f"\nDetox LOS summary:")
print(dxs['detox_los'].describe().round(2).to_string())
print(f"\nMissing LOS (no detox record): {dxs['detox_los'].isna().sum()}")
print()
dxs.head(80)

Dropped: ['DXLSOPTM', 'DXADMNTM']
DXS final shape: (658, 9)
DXS columns after rename:
['PATID', 'last_opioid_date', 'last_substance_code', 'last_route_code', 'last_opioid_prior_to_rand', 'detox_admission_day', 'detox_discharge_day', 'discharge_facility_code', 'followup_care_start_day', 'detox_los', 'last_substance_label', 'last_route_label', 'discharge_facility_label']
DXS clean shape: (658, 13)

Detox LOS summary:
count    569.00
mean       7.77
std        5.22
min        1.00
25%        5.00
50%        6.00
75%        9.00
max       40.00

Missing LOS (no detox record): 89



,PATID,last_opioid_date,last_substance_code,last_route_code,last_opioid_prior_to_rand,detox_admission_day,detox_discharge_day,discharge_facility_code,followup_care_start_day,detox_los,last_substance_label,last_route_label,discharge_facility_label
0,948442,0.0,5.0,1.0,NaN,-10.0,4.0,0.0,NaN,14.0,Non-study buprenorphine,Oral,Outpatient
1,85038,0.0,5.0,1.0,NaN,-3.0,15.0,0.0,NaN,18.0,Non-study buprenorphine,Oral,Outpatient
2,626492,0.0,5.0,1.0,NaN,-6.0,2.0,0.0,NaN,8.0,Non-study buprenorphine,Oral,Outpatient
3,632936,0.0,5.0,1.0,NaN,-4.0,4.0,0.0,NaN,8.0,Non-study buprenorphine,Oral,Outpatient
4,415576,0.0,5.0,1.0,NaN,-1.0,8.0,0.0,NaN,9.0,Non-study buprenorphine,Oral,Outpatient
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,504133,-12.0,5.0,1.0,1.0,-20.0,12.0,1.0,26.0,32.0,Non-study buprenorphine,Oral,Residential
76,884099,0.0,5.0,1.0,1.0,-14.0,6.0,1.0,15.0,20.0,Non-study buprenorphine,Oral,Residential
77,54463,-1.0,5.0,1.0,1.0,-14.0,7.0,0.0,NaN,21.0,Non-study buprenorphine,Oral,Outpatient
78,803304,-3.0,5.0,1.0,1.0,-9.0,7.0,0.0,NaN,16.0,Non-study buprenorphine,Oral,Outpatient


### Missing data  - DXS

In [7]:
# ── Missing data report for DXS ───────────────────────────────────────────────
missing_dxs = (
    dxs.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct_missing=lambda df: (df['n_missing'] / len(dxs) * 100).round(1))
    .sort_values('pct_missing', ascending=False)
)
print(f"DXS missing data report  ({len(dxs)} patients total)\n")
print(missing_dxs.to_string())

DXS missing data report  (658 patients total)

                           n_missing  pct_missing
followup_care_start_day          408         62.0
last_opioid_prior_to_rand        395         60.0
last_opioid_date                  90         13.7
detox_admission_day               89         13.5
detox_discharge_day               89         13.5
detox_los                         89         13.5
last_route_label                   9          1.4
last_substance_code                9          1.4
last_route_code                    9          1.4
last_substance_label               9          1.4
discharge_facility_code            5          0.8
discharge_facility_label           5          0.8
PATID                              0          0.0


In [9]:
# ── Load the naloxone challenge file ─────────────────────────────────────────
nxw = pd.read_excel(PATH_AE, sheet_name="nxw.csv")

# ── Identify the 89 patients with missing detox_admission_day ────────────────
missing_detox = set(dxs[dxs['detox_admission_day'].isna()]['PATID'])
print(f"Patients with missing detox_admission_day: {len(missing_detox)}")

# ── Hypothesis 1: are they concentrated in one treatment arm? ─────────────────
# For this we need the randomisation arm — it lives in the main outcomes file.
# As a proxy we check nxw.csv: XR-NTX patients who passed the naloxone
# challenge appear here; BUP-NX patients do not appear here at all.
nxw_ids = set(nxw['PATID'])

in_nxw     = missing_detox & nxw_ids        # had a naloxone challenge → XR-NTX arm
not_in_nxw = missing_detox - nxw_ids        # no naloxone challenge → likely BUP-NX

print(f"\nOf the {len(missing_detox)} patients with missing detox_admission_day:")
print(f"  Appear in nxw.csv (XR-NTX arm, passed naloxone challenge) : {len(in_nxw)}")
print(f"  Absent  from nxw.csv (likely BUP-NX or induction failure) : {len(not_in_nxw)}")

# ── Hypothesis 2: did they engage with the trial at all? ─────────────────────
# Check whether these patients appear in XRP (weekly relapse assessments)
# If they have XRP records they started treatment; if not they never engaged.
xrp = pd.read_csv("../raw-data/XRP.csv")
xrp_ids = set(xrp['PATID'])

engaged     = missing_detox & xrp_ids
not_engaged = missing_detox - xrp_ids

print(f"\n  Appear in XRP (engaged with treatment phase)  : {len(engaged)}")
print(f"  Absent  from XRP (never engaged / dropped out): {len(not_engaged)}")

# ── Hypothesis 3: did they have a recorded detox discharge day? ──────────────
# Some patients may have DXDDCDT but no DXADMNDT (discharge recorded, no admission)
has_discharge_no_admission = dxs[
    dxs['detox_admission_day'].isna() & dxs['detox_discharge_day'].notna()
]
print(f"\n  Have discharge day but no admission day: {len(has_discharge_no_admission)}")
print(f"  Missing both admission AND discharge day: "
      f"{dxs['detox_admission_day'].isna().sum() - len(has_discharge_no_admission)}")



Patients with missing detox_admission_day: 89

Of the 89 patients with missing detox_admission_day:
  Appear in nxw.csv (XR-NTX arm, passed naloxone challenge) : 0
  Absent  from nxw.csv (likely BUP-NX or induction failure) : 89

  Appear in XRP (engaged with treatment phase)  : 1
  Absent  from XRP (never engaged / dropped out): 88

  Have discharge day but no admission day: 0
  Missing both admission AND discharge day: 89


This means that the 89 missing values correspond to induction failures, thus, patients who do not engage with the trial. 

These patients who did not start the trial will only introduce 89 rows of complete *`Nan`* into the feature matrix. 

Their data can only be used for baseline tasks, hardly for prediction. 

##### *Drop screening failures*

In [10]:
# ── Drop patients with no detox record ───────────────────────────────────────
# These 89 patients have no admission day, no discharge day, no naloxone
# challenge, and no XRP engagement — consistent with screening failures
# who never entered the treatment phase. Verified against nxw and XRP.
before = len(dxs)
dxs = dxs[dxs['detox_admission_day'].notna()].copy()
print(f"Dropped {before - len(dxs)} patients with no detox record.")
print(f"DXS remaining: {len(dxs)} patients")

Dropped 89 patients with no detox record.
DXS remaining: 569 patients


Now, for those features *`last_opioid_prior_to_rand`* and *`followup_care_start_day`* that have more than 50% of missing values: 
- *`last_opioid_prior_to_rand`* only applicable for patients in the early randomisation group (randomised within 72h of las opioid use), that is why it is near 50%
- *`followup_care_start_day`* this was only filled when a patient transferred to a specific follow-up care facility after detox discharge, probably correlated with *`discharge_facility_code`*

In [11]:
# ── Drop features with >50% missingness and no independent information ────────
# last_opioid_prior_to_rand : only recorded for early-randomisation group (~50%)
# followup_care_start_day   : only recorded for residential-discharge patients
cols_to_drop = ['last_opioid_prior_to_rand', 'followup_care_start_day']
cols_to_drop = [c for c in cols_to_drop if c in dxs.columns]  # safe guard
dxs = dxs.drop(columns=cols_to_drop)
print(f"Dropped: {cols_to_drop}")
print(f"DXS shape after dropping: {dxs.shape}")

Dropped: ['last_opioid_prior_to_rand', 'followup_care_start_day']
DXS shape after dropping: (569, 11)


As for the rest of the missing values (only remain 4 features with 2 missing values), since they are all categorical, the mode will be imputed. 

In [12]:
# ── Verify it is the same 2 patients across all affected columns ──────────────
affected_cols = [
    'last_opioid_date', 'last_substance_code', 'last_route_code',
    'last_route_label', 'last_substance_label'
]
missing_mask = dxs[affected_cols].isnull().any(axis=1)
print(f"Patients with any missing value in these columns: {missing_mask.sum()}")
print(dxs.loc[missing_mask, ['PATID'] + affected_cols])

Patients with any missing value in these columns: 2
      PATID  last_opioid_date  last_substance_code  last_route_code  \
133  601547               NaN                  NaN              NaN   
181  516165               NaN                  NaN              NaN   

    last_route_label last_substance_label  
133              NaN                  NaN  
181              NaN                  NaN  


In [13]:
# ── Impute with mode ──────────────────────────────────────────────────────────
for col in affected_cols:
    mode_val = dxs[col].mode()[0]
    n_missing = dxs[col].isna().sum()
    dxs[col] = dxs[col].fillna(mode_val)
    print(f"  {col}: filled {n_missing} NaN(s) with mode → '{mode_val}'")

print(f"\nDXS missing values remaining: {dxs.isnull().sum().sum()}")

  last_opioid_date: filled 2 NaN(s) with mode → '-1.0'
  last_substance_code: filled 2 NaN(s) with mode → '1.0'
  last_route_code: filled 2 NaN(s) with mode → '1.0'
  last_route_label: filled 2 NaN(s) with mode → 'Oral'
  last_substance_label: filled 2 NaN(s) with mode → 'Methadone'

DXS missing values remaining: 0


### 2b. DSM 
The DSM file contains 11 binary criteria per substance (66 item columns total). For the baseline feature matrix we retain only the **summary severity scores** (`DS*SCO`, one per substance) plus the 11-item binary arrays for opioids (the primary diagnosis). The raw items for other substances can be added if needed, they are simply dropped here for parsimony.

In [14]:
# ── Confirm baseline-only (should all be PROTSEG=A, VISNO=0) ─────────────────
assert dsm_raw['PROTSEG'].eq('A').all(), "Unexpected non-baseline rows in DSM"
assert dsm_raw['VISNO'].eq(0).all() or dsm_raw['VISNO'].isin([0, -1]).all(), \
    "Unexpected VISNO values in DSM"

# ── Select summary scores (one per substance) and opioid item-level flags ─────
# Summary scores: DSOPISCO, DSALCSCO, DSAMPSCO, DSTHCSCO, DSCOCSCO, DSSEDSCO
score_cols = [c for c in dsm_raw.columns if c.endswith('SCO')]

# Opioid item-level DSM-5 criteria (all columns starting with DSOPI, excluding the score)
# These capture which specific criteria were met for the primary diagnosis
opioid_item_cols = [c for c in dsm_raw.columns
                    if c.startswith('DSOPI') and not c.endswith('SCO')]

dsm = dsm_raw[['PATID'] + score_cols + opioid_item_cols].copy()

# ── Human-readable score rename for clarity ───────────────────────────────────
score_rename = {
    'DSOPISCO': 'dsm_opioid_score',
    'DSALCSCO': 'dsm_alcohol_score',
    'DSAMPSCO': 'dsm_stimulant_score',
    'DSTHCSCO': 'dsm_cannabis_score',
    'DSCOCSCO': 'dsm_cocaine_score',
    'DSSEDSCO': 'dsm_sedative_score',
}
dsm = dsm.rename(columns=score_rename)

# ── Rename opioid item-level DSM-5 criteria columns ──────────────────────────
# Each item is binary (1 = criterion met) and corresponds to one of the
# 11 DSM-5 criteria for Opioid Use Disorder
dsm_item_rename = {
    'DSOPITOL' : 'opi_tolerance',            # 1. Tolerance (needing more for same effect)
    'DSOPIWIT' : 'opi_withdrawal',           # 2. Withdrawal symptoms when stopping
    'DSOPILRG' : 'opi_larger_amounts',       # 3. Taking larger amounts than intended
    'DSOPICTL' : 'opi_failed_cutdown',       # 4. Persistent desire / failed efforts to cut down
    'DSOPITME' : 'opi_time_spent',           # 5. Great deal of time obtaining / using / recovering
    'DSOPICRA' : 'opi_strong_craving',              # 6. Craving or strong urge to use
    'DSOPIRSP' : 'opi_role_failure',         # 7. Failure to fulfil major role obligations
    'DSOPISOC' : 'opi_social_problems',      # 8. Continued use despite social / interpersonal problems
    'DSOPIACT' : 'opi_activities_reduced',   # 9. Important activities given up or reduced
    'DSOPIHAZ' : 'opi_hazardous_use',        # 10. Recurrent use in physically hazardous situations
    'DSOPIPRB' : 'opi_despite_problems',
    'DSOPI12M':  'opi_past_12_months', 
    'DSOPIOBL': 'opi_recurrent_obligat_fail', 
    'DSOPIDOS': 'opi_larger_longer_dose', 
    'DSOPICUT' : 'opi_cut_back_unsuccess', 
    'DSOPITIM' : 'opi_time_spent',
    'DSOPICON' : 'opi_continued_use_despite_problems', 
}
dsm = dsm.rename(columns={k: v for k, v in dsm_item_rename.items() if k in dsm.columns})

print("DSM columns after rename:")
print(dsm.columns.tolist())

print(f"DSM clean shape: {dsm.shape}")
print(f"Unique PATIDs: {dsm['PATID'].nunique()}")
print(f"Duplicate PATIDs: {dsm['PATID'].duplicated().sum()}")
print()
print("Summary score means:")
score_renamed = list(score_rename.values())
print(dsm[score_renamed].mean().round(2).to_string())
print()
dsm.head(40)

DSM columns after rename:
['PATID', 'dsm_opioid_score', 'dsm_alcohol_score', 'dsm_stimulant_score', 'dsm_cannabis_score', 'dsm_cocaine_score', 'dsm_sedative_score', 'opi_past_12_months', 'opi_recurrent_obligat_fail', 'opi_hazardous_use', 'opi_social_problems', 'opi_tolerance', 'opi_withdrawal', 'opi_larger_longer_dose', 'opi_cut_back_unsuccess', 'opi_time_spent', 'opi_activities_reduced', 'opi_continued_use_despite_problems', 'opi_strong_craving']
DSM clean shape: (668, 19)
Unique PATIDs: 668
Duplicate PATIDs: 0

Summary score means:
dsm_opioid_score       1.02
dsm_alcohol_score      2.95
dsm_stimulant_score    2.67
dsm_cannabis_score     3.12
dsm_cocaine_score      2.46
dsm_sedative_score     2.76



,PATID,dsm_opioid_score,dsm_alcohol_score,dsm_stimulant_score,dsm_cannabis_score,dsm_cocaine_score,dsm_sedative_score,opi_past_12_months,opi_recurrent_obligat_fail,opi_hazardous_use,opi_social_problems,opi_tolerance,opi_withdrawal,opi_larger_longer_dose,opi_cut_back_unsuccess,opi_time_spent,opi_activities_reduced,opi_continued_use_despite_problems,opi_strong_craving
0,948442,1,2.0,4.0,3.0,2.0,1.0,1,0,0,1,1,1,1,1,1,1,1,1
1,85038,1,2.0,4.0,3.0,1.0,3.0,1,1,1,1,1,1,1,1,1,1,1,1
2,626492,1,4.0,4.0,4.0,4.0,4.0,1,1,1,1,1,1,1,1,1,1,1,1
3,632936,1,4.0,4.0,4.0,4.0,4.0,1,1,1,1,1,1,1,1,1,1,1,1
4,415576,1,4.0,4.0,4.0,4.0,4.0,1,1,1,1,1,1,1,1,1,1,1,1
5,471334,1,NaN,NaN,4.0,4.0,NaN,1,1,1,1,1,1,1,1,1,1,1,1
6,626802,1,NaN,NaN,NaN,NaN,NaN,1,1,1,1,1,1,1,1,1,1,1,1
7,422129,1,NaN,NaN,NaN,NaN,NaN,1,1,0,1,1,1,1,1,1,1,1,1
8,29321,1,4.0,NaN,4.0,NaN,NaN,1,1,1,1,1,1,1,1,1,1,1,1
9,873012,1,3.0,NaN,1.0,1.0,NaN,1,1,0,1,1,1,1,1,1,1,1,1


#### Missing data - DSM

In [15]:
# ── Missing data report for DXS ───────────────────────────────────────────────
missing_dsm = (
    dsm.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct_missing=lambda df: (df['n_missing'] / len(dsm) * 100).round(1))
    .sort_values('pct_missing', ascending=False)
)
print(f"DSM missing data report  ({len(dsm)} patients total)\n")
print(missing_dsm.to_string())

DSM missing data report  (668 patients total)

                                    n_missing  pct_missing
dsm_stimulant_score                       445         66.6
dsm_sedative_score                        323         48.4
dsm_cocaine_score                         319         47.8
dsm_cannabis_score                        232         34.7
dsm_alcohol_score                         231         34.6
dsm_opioid_score                            0          0.0
PATID                                       0          0.0
opi_past_12_months                          0          0.0
opi_recurrent_obligat_fail                  0          0.0
opi_hazardous_use                           0          0.0
opi_social_problems                         0          0.0
opi_tolerance                               0          0.0
opi_withdrawal                              0          0.0
opi_larger_longer_dose                      0          0.0
opi_cut_back_unsuccess                      0          0.0
opi_time_

##### What do `NaN` mean in this features? 

Each of the scores represents a severity tier (0-4) that is assessed against 11 criteria (e.g., tolerance, withdrawal, social problems...)
If we take a look at the opioid score, *there are no missing values*. This is relevant, because all of them have OUD (Opium Use Disorder). Hence, a `NaN` in another drug score, means that patient does not present a use disorder for that substance. 

Therefore, they will be imputed a score of 0 severity. 


In [16]:
# ── Impute NaN → 0 for substance severity scores ─────────────────────────────

score_cols = [
    'dsm_alcohol_score',
    'dsm_stimulant_score',
    'dsm_cannabis_score',
    'dsm_cocaine_score',
    'dsm_sedative_score',
]

for col in score_cols:
    n_imputed = dsm[col].isna().sum()
    dsm[col] = dsm[col].fillna(0).astype(int)
    print(f"  {col}: imputed {n_imputed} NaN(s) → 0")

print(f"\nDSM missing values remaining: {dsm.isnull().sum().sum()}")
print(f"\nUpdated score distributions:")
for col in score_cols:
    print(f"  {col}: {dsm[col].value_counts().sort_index().to_dict()}")

dsm['dsm_opioid_score'] = dsm['dsm_opioid_score'].astype(int)

  dsm_alcohol_score: imputed 231 NaN(s) → 0
  dsm_stimulant_score: imputed 445 NaN(s) → 0
  dsm_cannabis_score: imputed 232 NaN(s) → 0
  dsm_cocaine_score: imputed 319 NaN(s) → 0
  dsm_sedative_score: imputed 323 NaN(s) → 0

DSM missing values remaining: 0

Updated score distributions:
  dsm_alcohol_score: {0: 231, 1: 109, 2: 42, 3: 50, 4: 236}
  dsm_stimulant_score: {0: 445, 1: 71, 2: 27, 3: 30, 4: 95}
  dsm_cannabis_score: {0: 232, 1: 68, 2: 51, 3: 78, 4: 239}
  dsm_cocaine_score: {0: 319, 1: 143, 2: 32, 3: 43, 4: 131}
  dsm_sedative_score: {0: 323, 1: 104, 2: 33, 3: 50, 4: 158}


### 2c. ASE — filter to baseline visit and select analytic variables

We keep only `VISNO = '00'` (the screening/baseline visit). From the 65 available columns we select the variables with the clearest analytic meaning: education, occupation, days worked, income sources, and the total employment-days composite.

In [17]:
# ── Filter to baseline visit only ─────────────────────────────────────────────
ase_bl = ase_raw[ase_raw['VISNO'] == '00'].copy()

print(f"ASE baseline rows: {len(ase_bl)}")
print(f"Unique PATIDs:     {ase_bl['PATID'].nunique()}")
print(f"Duplicate PATIDs:  {ase_bl['PATID'].duplicated().sum()}")

# Handle rare duplicates (keep first record per patient if any exist)
if ase_bl['PATID'].duplicated().any():
    print("  → Deduplicating: keeping first record per patient")
    ase_bl = ase_bl.drop_duplicates(subset='PATID', keep='first')

# ── Select variables ──────────────────────────────────────────────────────────
# AEEDCPYR  = years of education completed
# AEOCCUPT  = current occupation category (1–7 scale)
# AEDRVLSC  = has a valid driver's licence (0/1)
# AEAUTOAV  = has an automobile available (0/1)
# AEPAID    = days of PAID employment, past 30 days
# AEEP30D   = days of ANY employment (paid or unpaid), past 30 days
# AEEMPMNY  = money from employment, past 30 days (dollars)
# AEUNEMNY  = money from unemployment benefits, past 30 days
# AEWLFMNY  = money from welfare, past 30 days
# AEILLMNY  = money from illegal activities, past 30 days
# AEDEPEND  = number of dependants financially supported

ase_cols = [
    'PATID',
    'AEEDCPYR',   # education years
    'AEOCCUPT',   # occupation category
    'AEDRVLSC',   # driver's licence
    'AEAUTOAV',   # automobile available
    'AEPAID',     # days paid employment
    'AEEP30D',    # days any employment
    'AEEMPMNY',   # income from employment ($)
    'AEUNEMNY',   # unemployment benefits ($)
    'AEWLFMNY',   # welfare ($)
    'AEILLMNY',   # illegal income ($)
    'AEDEPEND',   # number of dependants
]

# Keep only columns that actually exist in this version of the file
ase_cols_present = [c for c in ase_cols if c in ase_bl.columns]
missing_cols = set(ase_cols) - set(ase_cols_present)
if missing_cols:
    print(f"  Note: columns not found and skipped: {missing_cols}")

ase = ase_bl[ase_cols_present].copy()

# ── Human-readable renames ────────────────────────────────────────────────────
ase_rename = {
    'AEEDCPYR': 'education_years',
    'AEOCCUPT': 'occupation_category',
    'AEDRVLSC': 'has_drivers_licence',
    'AEAUTOAV': 'has_automobile',
    'AEPAID':   'days_paid_employment_30d',
    'AEEP30D':  'days_any_employment_30d',
    'AEEMPMNY': 'income_employment_30d',
    'AEUNEMNY': 'income_unemployment_30d',
    'AEWLFMNY': 'income_welfare_30d',
    'AEILLMNY': 'income_illegal_30d',
    'AEDEPEND': 'n_dependants',
}
ase = ase.rename(columns={k: v for k, v in ase_rename.items() if k in ase.columns})

print(f"\nASE clean shape: {ase.shape}")
print()
ase.head(40)

ASE baseline rows: 623
Unique PATIDs:     623
Duplicate PATIDs:  0

ASE clean shape: (623, 12)



,PATID,education_years,occupation_category,has_drivers_licence,has_automobile,days_paid_employment_30d,days_any_employment_30d,income_employment_30d,income_unemployment_30d,income_welfare_30d,income_illegal_30d,n_dependants
1,948442,10,7.0,0,0,0.0,30.0,0.0,0.0,213.0,300.0,0
3,85038,12,5.0,0,0,0.0,30.0,0.0,0.0,0.0,600.0,0
7,626492,12,7.0,0,0,0.0,4.0,0.0,0.0,347.0,400.0,1
10,632936,13,6.0,0,0,16.0,0.0,400.0,0.0,0.0,0.0,0
13,415576,12,5.0,1,1,0.0,30.0,0.0,0.0,0.0,200.0,0
15,471334,11,7.0,1,1,0.0,28.0,0.0,0.0,0.0,300.0,1
17,626802,11,6.0,0,0,0.0,30.0,0.0,0.0,0.0,2500.0,0
19,422129,12,7.0,1,1,0.0,30.0,0.0,0.0,0.0,0.0,0
21,29321,13,5.0,1,0,0.0,30.0,0.0,0.0,189.0,500.0,0
22,873012,12,4.0,0,0,0.0,0.0,0.0,0.0,189.0,0.0,0


In [18]:
# ── Missing data report for ASE ───────────────────────────────────────────────
missing_ase = (
    ase.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct_missing=lambda df: (df['n_missing'] / len(ase) * 100).round(1))
    .sort_values('pct_missing', ascending=False)
)
print(f"ASE missing data report  ({len(ase)} patients total)\n")
print(missing_ase.to_string())

ASE missing data report  (623 patients total)

                          n_missing  pct_missing
days_any_employment_30d          36          5.8
income_employment_30d             5          0.8
income_illegal_30d                5          0.8
occupation_category               1          0.2
has_drivers_licence               0          0.0
education_years                   0          0.0
PATID                             0          0.0
days_paid_employment_30d          0          0.0
has_automobile                    0          0.0
income_unemployment_30d           0          0.0
income_welfare_30d                0          0.0
n_dependants                      0          0.0


The missing value features correspond to numerical distributions. These are mostly right-skewed and present some outliers (there are patiens with incomes higher than 50 000 that do not reflect any trend). 
Hence it is a bit dangerous to impute the mean, we will impute the median. 

However, we see that the median is 0 (most of them don't have jobs or income), but that is an honest reflection of the cohort status. 

In [19]:
# ── Impute missing ASE values with column median ──────────────────────────────
# Missingness is low (<6%) and not structurally informative.
# Median preferred over mean due to right-skewed distributions and outliers.

ase_impute_cols = [
    'days_any_employment_30d',
    'income_employment_30d',
    'income_illegal_30d',
    'occupation_category',
]

for col in ase_impute_cols:
    median_val = ase[col].median()
    n_imputed  = ase[col].isna().sum()
    ase[col]   = ase[col].fillna(median_val)
    print(f"  {col}: imputed {n_imputed} NaN(s) → median = {median_val}")


print(f"\nASE missing values remaining: {ase.isnull().sum().sum()}")

  days_any_employment_30d: imputed 36 NaN(s) → median = 0.0
  income_employment_30d: imputed 5 NaN(s) → median = 0.0
  income_illegal_30d: imputed 5 NaN(s) → median = 0.0
  occupation_category: imputed 1 NaN(s) → median = 5.0

ASE missing values remaining: 0


## 3. Check coverage before merging

Before joining, we verify how many of the 658 DXS patients appear in each of the other two files. Patients absent from DSM or ASE will receive `NaN` for those variables after the left-join.

In [20]:
dxs_ids  = set(dxs['PATID'])
dsm_ids  = set(dsm['PATID'])
ase_ids  = set(ase['PATID'])

print("Coverage relative to DXS backbone (N=658):")
print(f"  DXS  patients:  {len(dxs_ids):>4}  (backbone)")
print(f"  DSM  patients:  {len(dsm_ids):>4}  "
      f"| in DXS: {len(dsm_ids & dxs_ids):>4} "
      f"| only in DSM: {len(dsm_ids - dxs_ids):>3} "
      f"| only in DXS: {len(dxs_ids - dsm_ids):>3}")
print(f"  ASE  patients:  {len(ase_ids):>4}  "
      f"| in DXS: {len(ase_ids & dxs_ids):>4} "
      f"| only in ASE: {len(ase_ids - dxs_ids):>3} "
      f"| only in DXS: {len(dxs_ids - ase_ids):>3}")
print()
print("Patients in all three files:", len(dxs_ids & dsm_ids & ase_ids))
print()
print("Note: patients in DSM or ASE but NOT in DXS are likely screening")
print("failures recorded before randomisation. They will be dropped by")
print("the left-join (DXS is the authoritative patient list).")

Coverage relative to DXS backbone (N=658):
  DXS  patients:   569  (backbone)
  DSM  patients:   668  | in DXS:  569 | only in DSM:  99 | only in DXS:   0
  ASE  patients:   623  | in DXS:  569 | only in ASE:  54 | only in DXS:   0

Patients in all three files: 569

Note: patients in DSM or ASE but NOT in DXS are likely screening
failures recorded before randomisation. They will be dropped by
the left-join (DXS is the authoritative patient list).


## 4. Merge

We use a **left join from DXS** so that all 658 participants are preserved, even those without DSM or ASE records. The join key is `PATID` throughout.

Patients absent from DSM or ASE receive `NaN` for those columns — this is expected (see coverage check above) and does **not** indicate a data error.

In [63]:
# ── Merge: inner join since all 569 DXS patients appear in both DSM and ASE ──
# Left join and inner join are equivalent here (verified in coverage check),
# but we use inner explicitly to document the intent and guard against
# any unexpected strays introduced by future data updates.

merged = (
    dxs
    .merge(dsm, on='PATID', how='inner')
    .merge(ase, on='PATID', how='inner')
)

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert merged['PATID'].duplicated().sum() == 0, "Duplicate PATIDs after merge"
assert len(merged) == 569, f"Expected 569 rows, got {len(merged)}"

print(f"Merged shape : {merged.shape}")
print(f"Patients kept: {len(merged)}  (dropped {658 - len(merged)} screening failures)")
print(f"Missing values remaining: {merged.isnull().sum().sum()}")

Merged shape : (569, 40)
Patients kept: 569  (dropped 89 screening failures)
Missing values remaining: 0


## 5. Post-merge diagnostics
### 5a. Column inventory by source

In [64]:
dxs_cols  = [c for c in merged.columns if c in dxs.columns]
dsm_cols  = [c for c in merged.columns if c in dsm.columns and c != 'PATID']
ase_cols_ = [c for c in merged.columns if c in ase.columns and c != 'PATID']

print(f"── DXS  ({len(dxs_cols)} cols) ──────────────────────────")
print("  " + ", ".join(dxs_cols))

print(f"\n── DSM  ({len(dsm_cols)} cols) ──────────────────────────")
print("  " + ", ".join(dsm_cols))

print(f"\n── ASE  ({len(ase_cols_)} cols) ──────────────────────────")
print("  " + ", ".join(ase_cols_))

print(f"\nTotal columns: {merged.shape[1]}  |  Total patients: {merged.shape[0]}")

── DXS  (11 cols) ──────────────────────────
  PATID, last_opioid_date, last_substance_code, last_route_code, detox_admission_day, detox_discharge_day, discharge_facility_code, detox_los, last_substance_label, last_route_label, discharge_facility_label

── DSM  (18 cols) ──────────────────────────
  dsm_opioid_score, dsm_alcohol_score, dsm_stimulant_score, dsm_cannabis_score, dsm_cocaine_score, dsm_sedative_score, opi_past_12_months, opi_recurrent_obligat_fail, opi_hazardous_use, opi_social_problems, opi_tolerance, opi_withdrawal, opi_larger_longer_dose, opi_cut_back_unsuccess, opi_time_spent, opi_activities_reduced, opi_continued_use_despite_problems, opi_strong_craving

── ASE  (11 cols) ──────────────────────────
  education_years, occupation_category, has_drivers_licence, has_automobile, days_paid_employment_30d, days_any_employment_30d, income_employment_30d, income_unemployment_30d, income_welfare_30d, income_illegal_30d, n_dependants

Total columns: 40  |  Total patients: 569


### 5b. Missingness report

In [65]:
missing = (
    merged.isnull().sum()
    .rename('n_missing')
    .to_frame()
    .assign(pct_missing=lambda df: (df['n_missing'] / len(merged) * 100).round(1))
    .query('n_missing > 0')
    .sort_values('pct_missing', ascending=False)
)

print(f"Columns with missing values: {len(missing)} of {merged.shape[1]}\n")
print(missing.to_string())

Columns with missing values: 0 of 40

Empty DataFrame
Columns: [n_missing, pct_missing]
Index: []


In [66]:
merged.to_csv("clean-data/pre-trial.csv", index=False)

print(f"Saved  →  clean-data/pre-trial.csv")
print(f"Shape  :  {merged.shape[0]} patients × {merged.shape[1]} variables")
print()
print("Columns written:")
for c in merged.columns:
    print(f"  {c}")

Saved  →  clean-data/pre-trial.csv
Shape  :  569 patients × 40 variables

Columns written:
  PATID
  last_opioid_date
  last_substance_code
  last_route_code
  detox_admission_day
  detox_discharge_day
  discharge_facility_code
  detox_los
  last_substance_label
  last_route_label
  discharge_facility_label
  dsm_opioid_score
  dsm_alcohol_score
  dsm_stimulant_score
  dsm_cannabis_score
  dsm_cocaine_score
  dsm_sedative_score
  opi_past_12_months
  opi_recurrent_obligat_fail
  opi_hazardous_use
  opi_social_problems
  opi_tolerance
  opi_withdrawal
  opi_larger_longer_dose
  opi_cut_back_unsuccess
  opi_time_spent
  opi_activities_reduced
  opi_continued_use_despite_problems
  opi_strong_craving
  education_years
  occupation_category
  has_drivers_licence
  has_automobile
  days_paid_employment_30d
  days_any_employment_30d
  income_employment_30d
  income_unemployment_30d
  income_welfare_30d
  income_illegal_30d
  n_dependants
